In [1]:
import pandas as pd
import numpy as np
import socket
import threading
import json
import random
import time
import pandas as pd
#tf = pd.read_csv('../input/edgeiiotset-cyber-security-dataset-of-iot-iiot/Edge-IIoTset dataset/Selected dataset for ML and DL/DNN-EdgeIIoT-dataset.csv', low_memory=False)
#tf=pd.read_csv('./Selected dataset for ML and DL/DNN-EdgeIIoT-dataset.csv', low_memory=False) 
tf=pd.read_csv('./Selected dataset for ML and DL/ML-EdgeIIoT-dataset.csv', low_memory=False)

In [2]:
tf.info()
print(tf['Attack_type'].value_counts())
#['Normal','DDoS_UDP','DDoS_ICMP','DDoS_TCP','DDoS_HTTP']
df = tf.loc[tf['Attack_type'].isin( ['Normal','DDoS_UDP','DDoS_TCP','DDoS_HTTP'])]#getting only ddos attack
print(df['Attack_type'].value_counts())
df.to_csv('DDoS_DNN.csv', encoding='utf-8', index=False)
df = pd.read_csv('./DDoS_DNN.csv', low_memory=False) ################save ddos data into another file
df
################data cleaning // shuffling //droping duplicates // cleaning null values#########################
from sklearn.utils import shuffle
drop_columns = ["frame.time", "ip.src_host", "ip.dst_host", "arp.src.proto_ipv4","arp.dst.proto_ipv4", 
                "http.file_data","http.request.full_uri","icmp.transmit_timestamp",
                "http.request.uri.query", "tcp.options","tcp.payload","tcp.srcport",
                "tcp.dstport", "udp.port", "mqtt.msg"]

df.drop(drop_columns, axis=1, inplace=True)
df.dropna(axis=0, how='any', inplace=True)
df.drop_duplicates(subset=None, keep="first", inplace=True)
df = shuffle(df)
df.isna().sum()
print(df['Attack_type'].value_counts())
####################################data preprocessing#######################################
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn import preprocessing

def encode_text_dummy(df, name):
    dummies = pd.get_dummies(df[name])
    for x in dummies.columns:
        dummy_name = f"{name}-{x}"
        df[dummy_name] = dummies[x]
    df.drop(name, axis=1, inplace=True)
    
encode_text_dummy(df,'http.request.method')
encode_text_dummy(df,'http.referer')
encode_text_dummy(df,"http.request.version")
encode_text_dummy(df,"dns.qry.name.len")
encode_text_dummy(df,"mqtt.conack.flags")
encode_text_dummy(df,"mqtt.protoname")
encode_text_dummy(df,"mqtt.topic")
########################saving preprocessed data to csv file##########################################
df.to_csv('preprocessed_DDOS_DNN.csv', encoding='utf-8', index=False)
df = pd.read_csv('./preprocessed_DDOS_DNN.csv', low_memory=False) 
df

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 157800 entries, 0 to 157799
Data columns (total 63 columns):
 #   Column                     Non-Null Count   Dtype  
---  ------                     --------------   -----  
 0   frame.time                 157800 non-null  object 
 1   ip.src_host                157800 non-null  object 
 2   ip.dst_host                157800 non-null  object 
 3   arp.dst.proto_ipv4         157800 non-null  object 
 4   arp.opcode                 157800 non-null  float64
 5   arp.hw.size                157800 non-null  float64
 6   arp.src.proto_ipv4         157800 non-null  object 
 7   icmp.checksum              157800 non-null  float64
 8   icmp.seq_le                157800 non-null  float64
 9   icmp.transmit_timestamp    157800 non-null  float64
 10  icmp.unused                157800 non-null  float64
 11  http.file_data             157800 non-null  object 
 12  http.content_length        157800 non-null  float64
 13  http.request.uri.query     15

,arp.opcode,arp.hw.size,icmp.checksum,icmp.seq_le,icmp.unused,http.content_length,http.response,http.tls_port,tcp.ack,tcp.ack_raw,...,dns.qry.name.len-_googlecast._tcp.local,mqtt.conack.flags-0,mqtt.conack.flags-0.0,mqtt.conack.flags-0x00000000,mqtt.protoname-0,mqtt.protoname-0.0,mqtt.protoname-MQTT,mqtt.topic-0,mqtt.topic-0.0,mqtt.topic-Temperature_and_Humidity
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,6.0,3.945129e+09,...,0,1,0,0,1,0,0,1,0,0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,15.0,2.686043e+09,...,0,0,0,1,1,0,0,1,0,0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,6.0,2.729505e+09,...,0,1,0,0,1,0,0,1,0,0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000e+00,...,0,0,1,0,0,1,0,0,1,0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,279.0,3.915393e+08,...,0,0,1,0,0,1,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59336,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000e+00,...,0,0,1,0,0,1,0,0,1,0
59337,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.601398e+09,...,0,1,0,0,1,0,0,1,0,0
59338,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,158787.0,2.371783e+09,...,0,1,0,0,1,0,0,1,0,0
59339,0.0,0.0,0.0,48352.0,0.0,0.0,0.0,0.0,0.0,0.000000e+00,...,0,0,1,0,0,1,0,0,1,0


In [3]:
feat_cols = list(df.columns)
label_col = "Attack_type"
feat_cols.remove(label_col)
skip_list = ["icmp.unused", "http.tls_port", "dns.qry.type", "mqtt.msg_decoded_as"]
df.drop(skip_list, axis=1, inplace=True)
feat_cols = list(df.columns)
feat_cols.remove(label_col)
X = df.drop([label_col], axis=1)
y = df[label_col]


#CollectNetworkTrafficData

# SendNetworkTrafficDataToCloud

In [4]:
def SendNetworkTrafficDataToCloud(data, tcp_ip, tcp_port):
    # Create a TCP/IP socket
    tcp_socket = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    print("TCP Server" )
    try:
        # Connect to the TCP server
        tcp_socket.connect((tcp_ip, tcp_port))
        
        # Send the data
        message = json.dumps(data)
        tcp_socket.sendall(message.encode('utf-8'))
        print(f"Sent NetworkTraffic DataFrame to TCP server at {tcp_ip}:{tcp_port} {message}")
    finally:
        tcp_socket.close()

# CollectNetworkTrafficData

In [5]:
# Main function to start UDP client and TCP listener
def main():

     # Get the IP address of the machine
    #hostname = socket.gethostname()
    #tcp_ip = socket.gethostbyname(hostname)
    tcp_ip='127.0.0.1'
    tcp_port = 12349      # Cloud server's TCP port for receiving data



    # Continuously send data to the edge server via UDP
    while True:
        index = random.randrange(0, len(X))  # Adjust range to fit your DataFrame size
        data = X.values[index].tolist()
        
        # Send the data to the edge server via UDP
        SendNetworkTrafficDataToCloud(data, tcp_ip, tcp_port)
        
        # Wait for 10 seconds before sending the next data
        time.sleep(10)

if __name__ == '__main__':
    main()

TCP Server
Sent NetworkTraffic DataFrame to TCP server at 127.0.0.1:12349 [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 59.0, 2337598480.0, 28846.0, 1.0, 0.0, 0.0, 0.0, 17.0, 1.0, 0.0, 5.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0]
TCP Server
Sent NetworkTraffic DataFrame to TCP server at 127.0.0.1:12349 [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 542314.0, 2372166995.0, 55311.0, 0.0, 0.0, 0.0, 0.0, 24.0, 1.0, 1440.0, 145121604.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0]
TCP Server
Sent NetworkTraffic DataFrame to TCP server at 127.0.0.1:12349 [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 56.0, 967825806.0, 22264.0, 0.0, 0.0, 0.0, 0.0, 16.0, 1.0, 0.0, 5.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0

TCP Server
Sent NetworkTraffic DataFrame to TCP server at 127.0.0.1:12349 [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1157225.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 1.0, 1.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0]
TCP Server
Sent NetworkTraffic DataFrame to TCP server at 127.0.0.1:12349 [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 15.0, 3569928.0, 62286.0, 0.0, 0.0, 0.0, 0.0, 16.0, 1.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0]
TCP Server
Sent NetworkTraffic DataFrame to TCP server at 127.0.0.1:12349 [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 6.0, 2901927906.0, 27061.0, 0.0, 0.0, 0.0, 0.0, 16.0, 1.0, 0.0, 59.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 1.

TCP Server
Sent NetworkTraffic DataFrame to TCP server at 127.0.0.1:12349 [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1751712980.0, 1751712980.0, 32336.0, 0.0, 0.0, 1.0, 0.0, 2.0, 0.0, 120.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 1.0, 1.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0]
TCP Server
Sent NetworkTraffic DataFrame to TCP server at 127.0.0.1:12349 [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 684447.0, 2372309128.0, 27361.0, 0.0, 0.0, 0.0, 0.0, 24.0, 1.0, 1440.0, 187674195.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0]
TCP Server
Sent NetworkTraffic DataFrame to TCP server at 127.0.0.1:12349 [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 56.0, 3483945546.0, 1610.0, 0.0, 0.0, 0.0, 0.0, 16.0, 1.0, 0.0, 5.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.

TCP Server
Sent NetworkTraffic DataFrame to TCP server at 127.0.0.1:12349 [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 2334941.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 1.0, 1.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0]
TCP Server
Sent NetworkTraffic DataFrame to TCP server at 127.0.0.1:12349 [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 6.0, 2820607616.0, 4261.0, 0.0, 0.0, 0.0, 0.0, 16.0, 1.0, 0.0, 59.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0]
TCP Server
Sent NetworkTraffic DataFrame to TCP server at 127.0.0.1:12349 [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 34984532.0, 1173836439.0, 25341.0, 0.0, 0.0, 0.0, 0.0, 16.0, 1.0, 0.0, 137473.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,

TCP Server
Sent NetworkTraffic DataFrame to TCP server at 127.0.0.1:12349 [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1705313.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 1.0, 1.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0]
TCP Server
Sent NetworkTraffic DataFrame to TCP server at 127.0.0.1:12349 [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 624.0, 3326380144.0, 13451.0, 0.0, 0.0, 0.0, 0.0, 16.0, 1.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 1.0, 1.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0]
TCP Server
Sent NetworkTraffic DataFrame to TCP server at 127.0.0.1:12349 [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 3187497384.0, 47552.0, 0.0, 0.0, 0.0, 0.0, 24.0, 1.0, 21.0, 1038.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 

TCP Server
Sent NetworkTraffic DataFrame to TCP server at 127.0.0.1:12349 [0.0, 0.0, 0.0, 38560.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 47007.0, 0.0, 1149111.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 1.0, 1.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0]
TCP Server
Sent NetworkTraffic DataFrame to TCP server at 127.0.0.1:12349 [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 12507.0, 0.0, 0.0, 1.0, 0.0, 2.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 1.0, 1.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0]
TCP Server
Sent NetworkTraffic DataFrame to TCP server at 127.0.0.1:12349 [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 2275452.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 1.0, 1.0, 0.0, 1

TCP Server
Sent NetworkTraffic DataFrame to TCP server at 127.0.0.1:12349 [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 3108698089.0, 22229.0, 0.0, 0.0, 0.0, 0.0, 24.0, 1.0, 24.0, 279.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 1.0, 1.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0]
TCP Server
Sent NetworkTraffic DataFrame to TCP server at 127.0.0.1:12349 [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 872526.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 1.0, 1.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0]
TCP Server
Sent NetworkTraffic DataFrame to TCP server at 127.0.0.1:12349 [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1655381.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 1.0, 1.0, 0.

TCP Server
Sent NetworkTraffic DataFrame to TCP server at 127.0.0.1:12349 [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 279.0, 1586342993.0, 44919.0, 0.0, 0.0, 0.0, 0.0, 16.0, 1.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 1.0, 1.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0, 0.0, 1.0, 0.0]
TCP Server


ConnectionRefusedError: [WinError 10061] No connection could be made because the target machine actively refused it